# nuReasoning planning tutorial

This tutorial covers 1) training the nuVLA model, 2) validating it, 3) evaluating planning on the validation split, and 4) a pointer to the official challenge submission (see `nureasoning_challenge_submission.ipynb`).

Prerequisites: the devkit environment (`pip install -e .`), a CUDA GPU, and the train/validation/test splits downloaded (see `nureasoning_data_visualization.ipynb`). The first code cell `chdir`s to the repository root so paths match the README (`./dataset`, `./nureasoning_vla_workspace`). Override them with the `NUREASONING_*` environment variables below. Training is opt-in because a full run takes hours.

nuVLA train/eval/benchmark flags use underscores (`--data_root`, `--checkpoint_dir`). Challenge submission flags use hyphens (`--data-root`, `--checkpoint-dir`).

## 1. Train nuVLA

The VLM backbone (Qwen3-VL) is supervised with reasoning text; the flow-matching DiT action expert is supervised with future ego trajectories. Use `torchrun --nproc_per_node=<N> -m nureasoning.nuvla.train` for multi-GPU training.

In [ ]:
import os
from pathlib import Path

_here = Path.cwd()
REPO_ROOT = _here if (_here / "nureasoning").is_dir() else _here.parent
os.chdir(REPO_ROOT)

TRAIN_ROOT = Path(os.environ.get("NUREASONING_TRAIN_ROOT", "./dataset/data/train"))
VALIDATION_ROOT = Path(os.environ.get("NUREASONING_VALIDATION_ROOT", "./dataset/data/validation"))
CHALLENGE_ROOT = Path(os.environ.get("NUREASONING_TEST_ROOT", os.environ.get("NUREASONING_CHALLENGE_ROOT", "./dataset/data/test")))
VLA_WORKSPACE = Path(os.environ.get("NUREASONING_VLA_WORKSPACE", "./nureasoning_vla_workspace"))
VLA_CHECKPOINT = Path(os.environ.get("NUREASONING_VLA_CHECKPOINT", str(VLA_WORKSPACE / "final")))
PLANNING_OUTPUT = Path(os.environ.get("NUREASONING_PLANNING_OUTPUT", str(VLA_WORKSPACE)))
VLM_MODEL = os.environ.get("NUREASONING_VLM_MODEL", "Qwen/Qwen3-VL-2B-Instruct")
MAX_CLIPS = int(os.environ.get("NUREASONING_MAX_CLIPS", "0"))
RUN_TRAINING = os.environ.get("NUREASONING_RUN_TRAINING", "0") == "1"

print("REPO_ROOT", REPO_ROOT)
print("TRAIN_ROOT", TRAIN_ROOT, "exists", TRAIN_ROOT.is_dir())
print("VALIDATION_ROOT", VALIDATION_ROOT, "exists", VALIDATION_ROOT.is_dir())
print("VLA_CHECKPOINT", VLA_CHECKPOINT, "exists", VLA_CHECKPOINT.is_dir())

if RUN_TRAINING:
    !python -m nureasoning.nuvla.train \
        --data_root "{TRAIN_ROOT}" \
        --test_data_root "{VALIDATION_ROOT}" \
        --vlm_model_path "{VLM_MODEL}" \
        --output_dir "{VLA_WORKSPACE}"
else:
    print("Skipping full nuVLA training. Set NUREASONING_RUN_TRAINING=1 to run it.")

## 2. Validate the model

Open-loop trajectory metrics (ADE / FDE / heading error) and reasoning-text generation on the validation split.

In [ ]:
if VLA_CHECKPOINT.is_dir():
    !python -m nureasoning.nuvla.evaluate \
        --checkpoint_dir "{VLA_CHECKPOINT}" \
        --test_data_root "{VALIDATION_ROOT}" \
        --mode both --visualize
else:
    print(f"Checkpoint not found: {VLA_CHECKPOINT}. Train first or set NUREASONING_VLA_CHECKPOINT.")

## 3. Planning benchmark (validation split, ground truth available)

Scores collision, driveable area, progress, comfort, and human likeness per clip. `--mode vla` (the default) loads the checkpoint; `--mode gt` scores the human trajectory as an oracle. Reports write to the workspace folder unless `NUREASONING_PLANNING_OUTPUT` is set.

A VQA-trained workspace (`nureasoning_vla_qa_workspace/final`) uses the same command — `--planning_prompt auto` then selects the scene-only prompt.

In [ ]:
if VLA_CHECKPOINT.is_dir():
    !python -m nureasoning.planning.benchmark \
        --data_root "{VALIDATION_ROOT}" \
        --mode vla \
        --checkpoint_dir "{VLA_CHECKPOINT}" \
        --max_clips {MAX_CLIPS} \
        --output_dir "{PLANNING_OUTPUT}"
else:
    print(f"Checkpoint not found: {VLA_CHECKPOINT}. Train first or set NUREASONING_VLA_CHECKPOINT.")

## 4. Generate the challenge submission (planning + reasoning)

The test split has 10 s of cameras + ego state through the key frame (no future trajectory) and answer-free `reasoning_questions.json` files. The official submission JSON is **one entry per clip**, each holding a trajectory and that clip's answers. See `docs/submission.md` and `nureasoning_challenge_submission.ipynb`.


In [ ]:
CHALLENGE_OUTPUT = Path(os.environ.get("NUREASONING_CHALLENGE_OUTPUT", "./challenge_submission.json"))

!python -m nureasoning.submission.challenge \
    --data-root "{CHALLENGE_ROOT}" \
    --provider split \
    --planning-provider constant_velocity \
    --reasoning-provider stub \
    --max-clips {MAX_CLIPS} \
    --output "{CHALLENGE_OUTPUT}"

For a full nuVLA submission, use the checkpoint-backed cell below. It is opt-in because loading the model and answering every challenge question is expensive; set `NUREASONING_RUN_FULL_SUBMISSION=1` when ready.

In [ ]:
RUN_FULL_SUBMISSION = os.environ.get("NUREASONING_RUN_FULL_SUBMISSION", "0") == "1"
if RUN_FULL_SUBMISSION and VLA_CHECKPOINT.is_dir():
    !python -m nureasoning.submission.challenge \
        --data-root "{CHALLENGE_ROOT}" \
        --provider nuvla \
        --checkpoint-dir "{VLA_CHECKPOINT}" \
        --max-clips {MAX_CLIPS} \
        --output "{CHALLENGE_OUTPUT}"
else:
    print("Skipping full nuVLA submission. Set NUREASONING_RUN_FULL_SUBMISSION=1 and provide a checkpoint to run it.")